<a href="https://colab.research.google.com/github/Adri22K/ProjetoAndreaBD/blob/colab/Testes_de_fontes_API.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# **Air Quality Meteo**

Imports necessários

## **Código Python**

Resultado de 120 linhas com 5 colunas

---


5 dias de resultados por hora

In [ ]:
!pip install openmeteo-requests
!pip install requests-cache retry-requests numpy pandas


   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 230.4/230.4 kB 10.9 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 817.7/817.7 kB 29.4 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 131.4/131.4 kB 10.3 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 396.0/396.0 kB 31.0 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 2.4/2.4 MB 65.8 MB/s eta 0:00:00
  Attempting uninstall: flatbuffers
    Found existing installation: flatbuffers 25.12.19
    Uninstalling flatbuffers-25.12.19:
      Successfully uninstalled flatbuffers-25.12.19
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 70.8/70.8 kB 2.2 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 73.1/73.1 kB 6.1 MB/s eta 0:00:00


In [ ]:
# @title
import openmeteo_requests

import pandas as pd
import requests_cache
from retry_requests import retry

# Setup the Open-Meteo API client with cache and retry on error
cache_session = requests_cache.CachedSession('.cache', expire_after = 3600)
retry_session = retry(cache_session, retries = 5, backoff_factor = 0.2)
openmeteo = openmeteo_requests.Client(session = retry_session)

# Make sure all required weather variables are listed here
# The order of variables in hourly or daily is important to assign them correctly below
url = "https://air-quality-api.open-meteo.com/v1/air-quality"
params = {
	"latitude": -23.5489,
	"longitude": -46.6388,
	"hourly": ["pm10", "pm2_5"],
	"domains": "cams_global",
}
responses = openmeteo.weather_api(url, params = params)

# Process first location. Add a for-loop for multiple locations or weather models
response = responses[0]
print(f"Coordinates: {response.Latitude()}°N {response.Longitude()}°E")
print(f"Elevation: {response.Elevation()} m asl")
print(f"Timezone difference to GMT+0: {response.UtcOffsetSeconds()}s")

# Process hourly data. The order of variables needs to be the same as requested.
hourly = response.Hourly()
hourly_pm10 = hourly.Variables(0).ValuesAsNumpy()
hourly_pm2_5 = hourly.Variables(1).ValuesAsNumpy()

hourly_data = {
	"date": pd.date_range(
		start = pd.to_datetime(hourly.Time(), unit = "s", utc = True),
		end =  pd.to_datetime(hourly.TimeEnd(), unit = "s", utc = True),
		freq = pd.Timedelta(seconds = hourly.Interval()),
		inclusive = "left"
	)
}

hourly_data["pm10"] = hourly_pm10
hourly_data["pm2_5"] = hourly_pm2_5

hourly_dataframe = pd.DataFrame(data = hourly_data)
print("\nHourly data\n", hourly_dataframe)

Coordinates: -23.5°N -46.59999084472656°E
Elevation: 738.0 m asl
Timezone difference to GMT+0: 0s

Hourly data
                          date       pm10      pm2_5
0   2026-08-24 00:00:00+00:00  10.000000   8.200000
1   2026-08-24 01:00:00+00:00   9.500000   7.900000
2   2026-08-24 02:00:00+00:00   9.200000   7.700000
3   2026-08-24 03:00:00+00:00   8.600000   7.200000
4   2026-08-24 04:00:00+00:00   9.300000   7.800000
..                        ...        ...        ...
115 2026-08-28 19:00:00+00:00  13.600000  13.300000
116 2026-08-28 20:00:00+00:00  18.000000  17.600000
117 2026-08-28 21:00:00+00:00  27.799999  27.200001
118 2026-08-28 22:00:00+00:00  36.099998  35.200001
119 2026-08-28 23:00:00+00:00  39.200001  38.200001

[120 rows x 3 columns]


## **IBGE**

Seleção de Dados Agregados

---

Os resultados podem mudar dependendo dos agredados selecionados, juntamente como periodo local, existem mais algumas configurações possiveis de colocar no link da API, juntamente com a base de identificadores para selecionar os agregados.

---

Abaixo um exemplo gerado por IA para apresentar os dados em uma tabela

In [ ]:
url_api = "https://servicodados.ibge.gov.br/api/v3/agregados/6579/periodos/2024/variaveis/9324?localidades=N1[all]"
print(f"URL configurada: {url_api}")

URL configurada: https://servicodados.ibge.gov.br/api/v3/agregados/6579/periodos/2024/variaveis/9324?localidades=N1[all]


In [ ]:
import requests
import pandas as pd

# Fazendo a requisição para a API do IBGE
response = requests.get(url_api)

if response.status_code == 200:
    dados_ibge = response.json()

    # A estrutura da API de Agregados do IBGE é complexa e aninhada.
    # Vamos extrair as informações principais para criar uma tabela legível.
    registros = []
    for variavel in dados_ibge:
        var_nome = variavel.get('variavel', 'N/A')
        unidade = variavel.get('unidade', 'N/A')
        for resultado in variavel.get('resultados', []):
            for serie in resultado.get('series', []):
                localidade = serie.get('localidade', {}).get('nome', 'Brasil')
                for periodo, valor in serie.get('serie', {}).items():
                    registros.append({
                        'Variável': var_nome,
                        'Unidade': unidade,
                        'Localidade': localidade,
                        'Período': periodo,
                        'Valor': valor
                    })

    # Criando o DataFrame
    df_ibge = pd.DataFrame(registros)
    display(df_ibge)
else:
    print(f"Erro ao acessar a API do IBGE: {response.status_code}")

,Variável,Unidade,Localidade,Período,Valor
0,População residente estimada,Pessoas,Brasil,2024,212583750


# **NASA POWE DAILY**

URL só apresenta resultados fazendo as pesquisa diretamente do Site, como JSON


*   T2M - Temperatura a 2 metros
*   PRECTOTCORR - Precipitação corrigida







In [ ]:
url_api = "https://power.larc.nasa.gov/api/temporal/daily/point?start=20150101&end=20160101&latitude=-23.54&longitude=-46.63&community=ag¶meters=T2M&header=true"
print(f"URL configurada: {url_api}")

URL configurada: https://power.larc.nasa.gov/api/temporal/daily/point?start=20150101&end=20160101&latitude=-23.54&longitude=-46.63&community=ag¶meters=T2M&header=true


# API - METEO

In [ ]:
https://air-quality-api.open-meteo.com/v1/air-quality?latitude=-23.64&longitude=-43.05&hourly=pm10,pm2_5,carbon_dioxide,nitrogen_dioxide,sulphur_dioxide,ozone&timezone=America%2FSao_Paulo&domains=cams_global

In [ ]:
import openmeteo_requests
import pandas as pd
import requests_cache

from retry_requests import retry

# Configuração do cliente
cache_session = requests_cache.CachedSession(
    ".cache",
    expire_after=3600
)

retry_session = retry(
    cache_session,
    retries=5,
    backoff_factor=0.2
)

openmeteo = openmeteo_requests.Client(
    session=retry_session
)

# API de qualidade do ar
url = "https://air-quality-api.open-meteo.com/v1/air-quality"

params = {
    "latitude": -23.64,
    "longitude": -43.05,
    "hourly": [
        "pm10",
        "pm2_5",
        "carbon_dioxide",
        "nitrogen_dioxide",
        "sulphur_dioxide",
        "ozone"
    ],
    "timezone": "America/Sao_Paulo",
    "domains": "cams_global",
    "forecast_days": 7
}

responses = openmeteo.weather_api(
    url,
    params=params
)

response = responses[0]
hourly = response.Hourly()

# Datas e horários fornecidos pela API
datas = pd.date_range(
    start=pd.to_datetime(
        hourly.Time(),
        unit="s",
        utc=True
    ),
    end=pd.to_datetime(
        hourly.TimeEnd(),
        unit="s",
        utc=True
    ),
    freq=pd.Timedelta(
        seconds=hourly.Interval()
    ),
    inclusive="left"
)

# Converte para o horário de São Paulo
datas = datas.tz_convert("America/Sao_Paulo")

# DataFrame com os dados horários
hourly_dataframe = pd.DataFrame({
    "data_hora": datas,
    "pm10": hourly.Variables(0).ValuesAsNumpy(),
    "pm2_5": hourly.Variables(1).ValuesAsNumpy(),
    "carbon_dioxide": hourly.Variables(2).ValuesAsNumpy(),
    "nitrogen_dioxide": hourly.Variables(3).ValuesAsNumpy(),
    "sulphur_dioxide": hourly.Variables(4).ValuesAsNumpy(),
    "ozone": hourly.Variables(5).ValuesAsNumpy()
})

# Cria uma coluna contendo somente a data
hourly_dataframe["data"] = (
    hourly_dataframe["data_hora"]
    .dt.date
)

# Calcula a média de cada poluente por dia
daily_dataframe = (
    hourly_dataframe
    .drop(columns="data_hora")
    .groupby("data", as_index=False)
    .mean(numeric_only=True)
    .round(2)
)

print("\nDados diários da qualidade do ar:\n")
print(daily_dataframe)


Dados diários da qualidade do ar:

         data       pm10  pm2_5  carbon_dioxide  nitrogen_dioxide  \
0  2026-08-24  21.940001  13.79      437.880005              0.58   
1  2026-08-25  21.870001  13.42      438.420013              0.53   
2  2026-08-26  15.230000   8.91      439.420013              0.73   
3  2026-08-27  15.560000   9.45      440.829987              1.49   
4  2026-08-28  26.650000  17.85      442.750000              3.28   
5  2026-08-29  27.629999  18.52             NaN              6.53   
6  2026-08-30        NaN    NaN             NaN               NaN   

   sulphur_dioxide      ozone  
0             0.66  74.709999  
1             0.54  71.669998  
2             0.43  69.120003  
3             0.53  70.080002  
4             1.52  79.879997  
5             1.48  62.000000  
6              NaN        NaN  
